In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.gold.outreach_message_variants
COMMENT 'AI-drafted appointment reminder variants. Messages are non-clinical and intended for simulated operational review only.'
AS
SELECT
  appointment_id,
  patient_pseudo_id,
  appointment_date,
  risk_segment,
  risk_score,
  'A' AS variant,
  'WARM_SUPPORTIVE' AS variant_style,
  ai_query(
    'databricks-llama-4-maverick',
    concat(
      'Draft a polite appointment reminder in English. ',
      'Maximum 45 words. ',
      'Do not provide medical advice. ',
      'Do not mention diagnoses, health conditions, medications, or risk reasons. ',
      'Do not include patient names or identifiers. ',
      'Appointment date: ', CAST(appointment_date AS STRING), '. ',
      'Tone: warm, supportive, respectful. ',
      'Ask the patient to reply YES to confirm attendance or contact the clinic to reschedule.'
    )
  ) AS message_text,
  'uc16_outreach_prompt_v1.0' AS prompt_version,
  current_timestamp() AS _generated_at
FROM poc_health_no_show.gold.outreach_prompt_sample;

INSERT INTO poc_health_no_show.gold.outreach_message_variants
SELECT
  appointment_id,
  patient_pseudo_id,
  appointment_date,
  risk_segment,
  risk_score,
  'B' AS variant,
  'CONCISE_ACTION_ORIENTED' AS variant_style,
  ai_query(
    'databricks-gpt-oss-20b',
    concat(
      'Draft a concise appointment reminder in English. ',
      'Maximum 30 words. ',
      'Do not provide medical advice. ',
      'Do not mention diagnoses, health conditions, medications, or risk reasons. ',
      'Do not include patient names or identifiers. ',
      'Appointment date: ', CAST(appointment_date AS STRING), '. ',
      'Tone: clear, direct, action-oriented. ',
      'Ask the patient to reply YES to confirm or call the clinic to reschedule.'
    )
  ) AS message_text,
  'uc16_outreach_prompt_v1.0' AS prompt_version,
  current_timestamp() AS _generated_at
FROM poc_health_no_show.gold.outreach_prompt_sample;

In [0]:
%sql
SELECT
  variant,
  COUNT(*) AS message_count,
  AVG(length(message_text)) AS avg_message_length
FROM poc_health_no_show.gold.outreach_message_variants
GROUP BY variant;

In [0]:
%sql
SELECT appointment_id, variant, message_text
FROM poc_health_no_show.gold.outreach_message_variants
LIMIT 10;

A/B tracking stretch goal

In [0]:
%sql
CREATE TABLE IF NOT EXISTS poc_health_no_show.gold.outreach_tracking (
  tracking_id STRING COMMENT 'Unique outreach tracking record',
  appointment_id STRING COMMENT 'Appointment being tracked',
  variant STRING COMMENT 'A or B message variant',
  channel STRING COMMENT 'Simulated outreach channel',
  sent_at TIMESTAMP COMMENT 'Simulated send timestamp',
  delivery_status STRING COMMENT 'Simulated delivery status',
  confirmation_status STRING COMMENT 'Simulated patient confirmation',
  attended_flag BOOLEAN COMMENT 'Simulated attendance outcome',
  created_at TIMESTAMP COMMENT 'Record creation timestamp'
)
COMMENT 'Simulated A/B outreach tracking table. No real messages are sent in this POC.';

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")